<a class="anchor" id="2">

# Notebook 5: Streaming
</a>

### Group 16 Members:

- Ana Margarida Macedo (20250405)
- Catarina Aboim (20250375)
- Margarida Craveiro (20250346)
- Matilde Simões (20250382)
- Lourenço Silva (20250453)

## Structured Streaming: Real-Time NYC Yellow Taxi Analytics

In this notebook, we apply **Spark Structured Streaming** to the NYC Yellow Taxi dataset, using **Apache Kafka** as both the input source and output sink — exactly as in a production real-time pipeline.

The pipeline follows the 5-step recipe from the lectures:

```
readStream (Kafka) → parse → withWatermark + window + groupBy → writeStream (Kafka) + checkpoint
```

Since no live Kafka broker is available on Lightning, we first **produce** the taxi recordsinto a Kafka topic using a background producer thread (simulating a real taxi GPS feed), and then **consume** and aggregate them with Structured Streaming, writing results back to a second Kafka topic.

<a class="anchor" id="1">

# **1. Overview & Architecture**
</a>

Before starting this approach is important to give a small context to remember the dataset chosen for this part:

The data is the **NYC Yellow Taxi Trip Records** cleaned and saved in Notebook 1 (`data/clean/taxi_clean.parquet`). Each record represents one completed trip.

| Column | Description |
|---|---|
| `tpep_pickup_datetime` | Trip start — used as **event time** |
| `tpep_dropoff_datetime` | Trip end |
| `PULocationID` | Pickup zone — used for demand aggregations |
| `VendorID` | Vendor identifier |
| `fare_amount` | Base fare (USD) |
| `tip_amount` | Tip (USD) |
| `trip_distance` | Distance (miles) |
| `total_amount` | Total charge (USD) |

Because no live taxi GPS feed is available, we **simulate** one by reading the Parquet file in a background thread and **producing** each trip as a JSON message into the `taxi-trips` Kafka topic — exactly as a real taxi meter would push events.

A separate producer script (`taxi_producer.py`) reads the cleaned taxi Parquet file and publishes events to two Kafka topics:

| Topic | Cadence | Payload |
|---|---|---|
| `trips` | every ~5 s (50 events/cycle) | `pickup_ts`, `dropoff_ts`, `PULocationID`, `DOLocationID`, `passenger_count`, `trip_distance`, `fare_amount`, `tip_amount`, `total_amount`, `payment_type` |
| `long_trips` | subset — only trips > 10 miles | same payload, filtered |

We use them to demonstrate the standard streaming patterns:

1. Reading a Kafka topic as a typed streaming DataFrame
2. Sinks and output modes (`append`, `update`, `complete`)
3. Windowed aggregations with watermarks
4. Stream-stream join: correlate `trips` with `long_trips` to surface high-fare long-haul events

Structured Streaming treats a stream as a table that updates over time, so the same DataFrame API works on both bounded and unbounded data.

<a class="anchor" id="2">

## **1.2. Dataset Context**
</a>
The data used in this notebook is the **NYC Yellow Taxi Trip Records**, previously cleaned and consolidated in Notebook 1. It covers four months 
of trips (January 2015 and January–March 2016) and is stored locally at `data/clean/taxi_clean.parquet`.

Each record represents a single completed trip and contains the following key attributes for streaming analysis:

| Column | Description |
|---|---|
| `tpep_pickup_datetime` | Trip start timestamp — used as **event time** |
| `tpep_dropoff_datetime` | Trip end timestamp |
| `PULocationID` | Pickup zone — used for demand aggregations |
| `VendorID` | Vendor identifier — used for revenue monitoring |
| `fare_amount` | Base fare in USD |
| `tip_amount` | Tip in USD |
| `trip_distance` | Distance in miles |
| `total_amount` | Total charge in USD |

Since no live data feed is available, simulate the stream was necessary, by reading the Parquet files incrementally using `maxFilesPerTrigger=1`, so that Spark processes one partition per micro-batch — mimicking trips arriving in real time. In production, this source would be replaced by a **Kafka topic** receiving live GPS events from taxi meters.

<a class="anchor" id="2">

# **2. Environment Setup**
</a>

In [ ]:
!pip install "pyspark==3.5.0"

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, LongType, TimestampType,
)

KAFKA_BOOTSTRAP_SERVERS = "localhost:8098"

In [ ]:
spark = (
    SparkSession.builder
    .appName("kafka_streaming")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]")
    .getOrCreate()
)

<a class="anchor" id="2">

# **2. Reading the trips stream**

(transform the database)
</a>

We subscribe to the **`trips`** topic, declare the JSON schema, and parse the Kafka **`value`** column into a typed DataFrame. The result, **`trips_df`**, is the canonical streaming DataFrame we´ll reuse throughout the rest the notebook.

In [ ]:
# Define the schema for the incoming trip data
trip_schema = StructType([
    StructField("tpep_pickup_datetime",  StringType(),  True),
    StructField("tpep_dropoff_datetime", StringType(),  True),
    StructField("PULocationID",          IntegerType(), True),
    StructField("DOLocationID",          IntegerType(), True),
    StructField("passenger_count",       IntegerType(), True),
    StructField("trip_distance",         DoubleType(),  True),
    StructField("fare_amount",           DoubleType(),  True),
    StructField("tip_amount",            DoubleType(),  True),
    StructField("total_amount",          DoubleType(),  True),
    StructField("payment_type",          IntegerType(), True),
])

# Read the raw trip data from the Kafka topic "trips"
raw_trips = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "trips")
    .option("startingOffsets", "earliest")
    .load()
)

# Parse the JSON value into a typed DataFrame and perform basic cleaning
trips_df = (
    raw_trips
    .select(F.from_json(F.col("value").cast("string"), trip_schema).alias("v"))
    .select(
        F.to_timestamp(F.col("v.tpep_pickup_datetime"),  "yyyy-MM-dd HH:mm:ss").alias("pickup_ts"),
        F.to_timestamp(F.col("v.tpep_dropoff_datetime"), "yyyy-MM-dd HH:mm:ss").alias("dropoff_ts"),
        F.col("v.PULocationID")    .alias("PULocationID"),
        F.col("v.DOLocationID")    .alias("DOLocationID"),
        F.col("v.passenger_count") .alias("passenger_count"),
        F.col("v.trip_distance")   .alias("trip_distance"),
        F.col("v.fare_amount")     .alias("fare_amount"),
        F.col("v.tip_amount")      .alias("tip_amount"),
        F.col("v.total_amount")    .alias("total_amount"),
        F.col("v.payment_type")    .alias("payment_type"),
    )
    .filter(F.col("pickup_ts").isNotNull())
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
)

Like the batch API, streaming operations are lazy - nothing happens until and call **`.start()`**.

<a class="anchor" id="2">

# **3. Sinks and output models**

(choose sink + output mode)
</a>

Structured Streaming supports several sinks (what deciding where we want our output to go (be stored)), we can have **Kafka**, **file** (parquet/JSON/CSV), foreach custom logic, **console** for debugging, and **memory** for interactive inspection from SQL. 

Each query also has an *output mode* that controls what gets written on each trigger:

- `append` — only new rows since the last trigger
- `update` — only rows whose aggregate value changed
- `complete` — the entire result table (only valid for aggregates)

For this project it was decided the **memory** sink with a query name, which lets us run ad-hoc Spark SQL against the rolling stream. For this case **append** is use as output mode because, one trip ......

In [ ]:
trips_query = (
    trips_df.writeStream
    .outputMode("append")
    .queryName("trips_table")
    .format("memory")
    .start()
)

In [ ]:

spark.sql("""
    SELECT PULocationID,
           COUNT(*)                    AS trip_count,
           ROUND(AVG(fare_amount), 2)  AS avg_fare,
           ROUND(AVG(trip_distance),2) AS avg_distance,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM trips_table
    GROUP BY PULocationID
    ORDER BY trip_count DESC
""").show()

<a class="anchor" id="6">

# **4. Windowed aggregations and watermarks**
</a>

Time-windowed aggregations allow to aggregate data over a sliding or tumbling time period rather than calculating them across the entire streaming. 

A **watermark** tells the engine how late an event can arrive and still be included in its window.
State for windows older than `max(event_time) − watermark` is dropped, which bounds the memory footprint of a long-running query.

Here we use a **3-minute tumbling window** on `pickup_ts` with a **30-second watermark** to compute per-location OHLC-style fare statistics.


In [ ]:
windowed = (
    trips_df
    .withWatermark("pickup_ts", "30 seconds")
    .groupBy(
        F.window("pickup_ts", "3 minutes"),
        "PULocationID",
    )
    .agg(
        F.count("*")                       .alias("trip_count"),
        F.avg("fare_amount")               .alias("avg_fare"),
        F.min("fare_amount")               .alias("min_fare"),
        F.max("fare_amount")               .alias("max_fare"),
        F.sum("total_amount")              .alias("total_revenue"),
        F.avg("trip_distance")             .alias("avg_distance"),
    )
)

windowed_query = (
    windowed.writeStream
    .outputMode("append")
    .queryName("trips_aggregated")
    .format("memory")
    .start()
)

In [ ]:
spark.sql("SELECT * FROM trips_aggregated ORDER BY window").show(truncate=False)

<a class="anchor" id="6">

# **5. Stream-stream join**
</a>

We join the `trips` stream with the `long_trips` stream to answer:
*for each long-haul trip (> 10 miles), does a matching high-fare event (fare > $30) appear on the same pickup location within 2 minutes?*

This mirrors the class lab's news→trade-reaction join pattern:

Three design points worth flagging:

1. **Both sides must be watermarked.** Stream-stream joins require a watermark on each side so Spark can bound the join state and eventually emit results.
2. **Tight watermarks.** Since `pickup_ts` is the original trip timestamp (not wall clock), events from the producer arrive with some replay lag. We use `30 seconds` on both sides.
3. **Interval condition.** The join uses a `BETWEEN` on the pickup timestamp, making this a *time-bounded* join with finite state.

In [ ]:
long_trip_schema = trip_schema  # identical schema, just a filtered topic

raw_long = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "long_trips")
    .option("startingOffsets", "earliest")
    .load()
)

# Note: event_time is the original trip pickup timestamp.
# not the article's published time. RSS publication is too low-frequency
# to align with the live trade stream.
long_trips_df = (
    raw_long
    .select(F.from_json(F.col("value").cast("string"), long_trip_schema).alias("v"))
    .select(
        F.to_timestamp(F.col("v.tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss").alias("pickup_ts"),
        F.col("v.PULocationID") .alias("PULocationID"),
        F.col("v.trip_distance").alias("trip_distance"),
        F.col("v.fare_amount")  .alias("fare_amount"),
        F.col("v.total_amount") .alias("total_amount"),
    )
    .filter(F.col("pickup_ts").isNotNull())
)

In [ ]:
# Watermark each side. Since `event_time` is the producer's
# observation timestamp (always 'now') for both streams, events
# are never more than a few seconds late - so the watermarks can
# be tight. 
trips_wm     = trips_df.withWatermark("pickup_ts", "30 seconds").alias("t") # t stands for "trips"
long_trips_wm = long_trips_df.withWatermark("pickup_ts", "30 seconds").alias("l") # l stands for "long_trips"

# For each long trip, find any trip on the same pickup location
# with fare > $30 that fired within the next 2 minutes.
joined = trips_wm.join(
    long_trips_wm,
    F.expr("""
        t.PULocationID = l.PULocationID AND
        t.fare_amount > 30             AND
        t.pickup_ts BETWEEN l.pickup_ts AND l.pickup_ts + interval 2 minutes
    """),
)

# Roll up to one row per long trip — how many high-fare matches appeared?
reaction = (
    joined.groupBy("l.PULocationID", "l.pickup_ts", "l.trip_distance")
    .agg(
        F.count("*")             .alias("matching_high_fare_trips"),
        F.first("t.fare_amount") .alias("first_high_fare"),
        F.max("t.fare_amount")   .alias("max_fare_in_window"),
    )
)

reaction_query = (
    reaction.writeStream
    .outputMode("update")
    .queryName("long_trip_alerts")
    .format("memory")
    .start()
)

**Expect the first rows to appear ~3–4 minutes after starting the producer.** Stream-stream joins only support `append` output mode, meaning Spark holds each window's output until the watermark has passed the window's end.

In [ ]:
spark.sql("""
    SELECT PULocationID,
           pickup_ts,
           trip_distance,
           matching_high_fare_trips,
           ROUND(first_high_fare, 2)     AS first_high_fare,
           ROUND(max_fare_in_window, 2)  AS max_fare_in_window
    FROM long_trip_alerts
    ORDER BY pickup_ts DESC
""").show(truncate=False)

<a class="anchor" id="2">

# **7. Stop Queries and Spark Session**
</a>

In [ ]:
for q in [trips_query, windowed_query, reaction_query]:
    q.stop()

In [ ]:
spark.stop()

<a class="anchor" id="2">

# **8. Commands how to run it**
</a>

